In [ ]:
# ==============================
# AUTO bbox3 + Rotation Squeeze (FAST, Auto-best)
# - Runs bbox3 with random/SA-like sampling
# - Applies "fix_direction" rotation optimization (NO shapely polygons)
# - Tracks BEST automatically and writes /kaggle/working/submission.csv
# ==============================

import os, shutil, subprocess, random, time, math, tempfile
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull
from scipy.optimize import minimize_scalar

# ------------------------------
# PATHS
# ------------------------------
START_SUB  = "/kaggle/input/intergration-of-existing-result-current-best/submission.csv"   # or your baseline submission
START_BBOX = "/kaggle/input/simulated-annealing-manual-fun-tree-mover/bbox3"

WORK_SUB = "/kaggle/working/submission.csv"
BBOX     = "/kaggle/working/bbox3"

os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.exists(WORK_SUB):
    shutil.copy(START_SUB, WORK_SUB)
if not os.path.exists(BBOX):
    shutil.copy(START_BBOX, BBOX)
os.chmod(BBOX, 0o755)

# ------------------------------
# BUDGET / SEARCH CONFIG
# ------------------------------
MAX_SECONDS = 60 * 60 * 2   # 2 hours, change as needed
PER_RUN_TIMEOUT = 1200      # 20 min bbox3 timeout (per run)

# bbox3 parameter ranges
N_RANGE_WIDE = (1000, 2200)
R_RANGE_WIDE = (10, 100)

# Rotation optimization settings
ROTATE_GROUPS_FROM = 200    # rotate optimize groups 200..3
ROTATE_GROUPS_TO   = 3
ROTATE_EVERY_ACCEPTED = 1   # apply rotation on every accepted candidate; set >1 to reduce time

# Acceptance (simple SA)
USE_SA = True
T0 = 1e-3
T_MIN = 1e-6
ALPHA = 0.995

# ------------------------------
# Tree template points (15 vertices) in original coords
# same geometry as common Santa 2025 solutions
# ------------------------------
TREE_PTS = np.array([
    (0,0.8),(0.125,0.5),(0.0625,0.5),(0.2,0.25),(0.1,0.25),(0.35,0),
    (0.075,0),(0.075,-0.2),(-0.075,-0.2),(-0.075,0),(-0.35,0),
    (-0.1,0.25),(-0.2,0.25),(-0.0625,0.5),(-0.125,0.5)
], dtype=np.float64)

# ------------------------------
# Fast parsing / fast score (no overlap check)
# score = sum_{n=1..200} (side_n^2 / n)
# side_n is max(width,height) of all rotated polygons in group n
# ------------------------------
def parse_submission(path: str):
    df = pd.read_csv(path, usecols=["id","x","y","deg"], dtype=str)
    grp = df["id"].str.slice(0,3).astype(np.int16).to_numpy()
    x   = df["x"].str.lstrip("sS").astype(np.float64).to_numpy()
    y   = df["y"].str.lstrip("sS").astype(np.float64).to_numpy()
    deg = df["deg"].str.lstrip("sS").astype(np.float64).to_numpy()
    return df, grp, x, y, deg

def group_points(xi, yi, degi):
    """Return stacked points (m*P,2) for a group"""
    rad = np.deg2rad(degi)
    c = np.cos(rad)[:,None]
    s = np.sin(rad)[:,None]
    px = TREE_PTS[:,0][None,:]
    py = TREE_PTS[:,1][None,:]
    X = px*c - py*s + xi[:,None]
    Y = px*s + py*c + yi[:,None]
    pts = np.stack([X, Y], axis=-1).reshape(-1,2)
    return pts

def eval_fast_score(grp, x, y, deg):
    total = 0.0
    for n in range(1, 201):
        idx = np.where(grp == n)[0]
        if idx.size == 0:
            continue
        pts = group_points(x[idx], y[idx], deg[idx])
        mn = pts.min(axis=0); mx = pts.max(axis=0)
        side = max(mx[0]-mn[0], mx[1]-mn[1])
        total += (side*side)/n
    return float(total)

# ------------------------------
# Rotation optimization ("fix_direction") WITHOUT shapely
# - find angle in (0..90) minimizing bbox side of convex hull points
# - then rotate all centers around group bbox center, add angle to deg
# ------------------------------
def bbox_side_at_angle(angle_deg, hull_pts):
    a = np.deg2rad(angle_deg)
    c, s = np.cos(a), np.sin(a)
    # rotate points by -a (equivalent to evaluating bbox under +a)
    R = np.array([[c, s], [-s, c]], dtype=np.float64)
    rp = hull_pts @ R.T
    mn = rp.min(axis=0); mx = rp.max(axis=0)
    return max(mx[0]-mn[0], mx[1]-mn[1])

def optimize_rotation_for_points(pts):
    # convex hull
    if pts.shape[0] <= 3:
        hull_pts = pts
    else:
        hull = ConvexHull(pts)
        hull_pts = pts[hull.vertices]

    side0 = bbox_side_at_angle(0.0, hull_pts)
    res = minimize_scalar(lambda a: bbox_side_at_angle(a, hull_pts),
                          bounds=(0.001, 89.999), method="bounded")
    side1 = res.fun
    ang1  = res.x

    if side1 + 1e-9 < side0:
        return float(side1), float(ang1)
    else:
        return float(side0), 0.0

def apply_group_rotation(xg, yg, degg, pts, angle_deg):
    if abs(angle_deg) < 1e-12:
        return xg, yg, degg

    mn = pts.min(axis=0); mx = pts.max(axis=0)
    center = (mn + mx) / 2.0

    a = np.deg2rad(angle_deg)
    c, s = np.cos(a), np.sin(a)
    R = np.array([[c, -s],[s, c]], dtype=np.float64)

    centers = np.stack([xg, yg], axis=1)
    shifted = centers - center[None,:]
    rot = shifted @ R.T + center[None,:]

    return rot[:,0], rot[:,1], degg + angle_deg

def fix_direction_arrays(grp, x, y, deg,
                         n_from=200, n_to=3):
    """
    Apply rotation squeeze for groups n_from..n_to (descending).
    Return updated (x,y,deg). Only updates when side improves.
    """
    for n in range(n_from, n_to-1, -1):
        idx = np.where(grp == n)[0]
        if idx.size == 0:
            continue
        pts = group_points(x[idx], y[idx], deg[idx])
        mn = pts.min(axis=0); mx = pts.max(axis=0)
        side0 = max(mx[0]-mn[0], mx[1]-mn[1])

        best_side, best_ang = optimize_rotation_for_points(pts)
        if best_ang != 0.0 and best_side + 1e-9 < side0:
            # apply rotation to centers/angles
            x_new, y_new, deg_new = apply_group_rotation(
                x[idx], y[idx], deg[idx], pts, best_ang
            )
            x[idx] = x_new
            y[idx] = y_new
            deg[idx] = deg_new
    return x, y, deg

def write_submission_like(df_template, x, y, deg, out_path):
    # keep ids; ensure 's' prefix
    out = pd.DataFrame({
        "id": df_template["id"].astype(str),
        "x":  ["s"+repr(v) for v in x],
        "y":  ["s"+repr(v) for v in y],
        "deg":["s"+repr(v) for v in deg],
    })
    out.to_csv(out_path, index=False)

# ------------------------------
# bbox3 runner in temp dir
# ------------------------------
def run_bbox3_on_submission(best_sub_path, n, r):
    tmp = tempfile.mkdtemp(prefix="bbox3tmp_")
    sub = os.path.join(tmp, "submission.csv")
    shutil.copy(best_sub_path, sub)

    try:
        subprocess.run([BBOX, "-n", str(n), "-r", str(r)],
                       cwd=tmp,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL,
                       timeout=PER_RUN_TIMEOUT,
                       check=False)
        return tmp, sub
    except subprocess.TimeoutExpired:
        return tmp, None

def cleanup(tmp):
    try:
        shutil.rmtree(tmp, ignore_errors=True)
    except:
        pass

# ------------------------------
# MAIN LOOP (auto best)
# ------------------------------
start_time = time.time()
deadline = start_time + MAX_SECONDS

df0, grp0, x0, y0, deg0 = parse_submission(WORK_SUB)
best_score = eval_fast_score(grp0, x0, y0, deg0)
best_path = WORK_SUB

print(f"Start score = {best_score:.12f}")

T = T0
iters = 0
accepted = 0
best_updates = 0

# keep current as best (simple)
cur_path = best_path
cur_score = best_score

while time.time() < deadline:
    iters += 1

    # sample params
    n = random.randint(*N_RANGE_WIDE)
    r = random.randint(*R_RANGE_WIDE)

    tmpdir, cand_path = run_bbox3_on_submission(cur_path, n, r)
    try:
        if cand_path is None:
            continue

        # fast score candidate
        dfc, grpc, xc, yc, degc = parse_submission(cand_path)
        cand_score = eval_fast_score(grpc, xc, yc, degc)

        delta = cand_score - cur_score
        if USE_SA:
            if delta < 0:
                accept = True
            else:
                accept = (random.random() < math.exp(-delta / max(T, 1e-12)))
        else:
            accept = (delta < 0)

        if accept:
            accepted += 1

            # optional rotation squeeze (cheap version, no shapely)
            if (accepted % ROTATE_EVERY_ACCEPTED) == 0:
                xc2 = xc.copy(); yc2 = yc.copy(); degc2 = degc.copy()
                xc2, yc2, degc2 = fix_direction_arrays(
                    grpc, xc2, yc2, degc2,
                    n_from=ROTATE_GROUPS_FROM, n_to=ROTATE_GROUPS_TO
                )
                cand_score2 = eval_fast_score(grpc, xc2, yc2, degc2)

                if cand_score2 < cand_score:
                    # overwrite candidate file with rotated version
                    write_submission_like(dfc, xc2, yc2, degc2, cand_path)
                    cand_score = cand_score2

            # update current
            cur_score = cand_score
            # save current stable
            stable_cur = tempfile.NamedTemporaryFile(delete=False, suffix=".csv").name
            shutil.copy(cand_path, stable_cur)
            cur_path = stable_cur

            # update best
            if cand_score < best_score - 1e-12:
                best_score = cand_score
                best_updates += 1
                shutil.copy(cand_path, WORK_SUB)
                best_path = WORK_SUB
                print(f"🏆 BEST {best_updates} @ iter {iters}: {best_score:.12f} (n={n}, r={r})")

        # anneal
        if USE_SA:
            T = max(T * ALPHA, T_MIN)

        if iters % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Iter {iters:6d} | {elapsed:7.1f}s | T={T:.2e} | cur={cur_score:.12f} | best={best_score:.12f} | acc={accepted}")

    finally:
        cleanup(tmpdir)

print(f"\nDONE. Best score = {best_score:.12f}")
print(f"Saved to {WORK_SUB}")
